# U2-02 · Salud, crédito y ventas: medidas y modelos lineales

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 2, apartados 2.1 a 2.3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar scikit-learn.

> Datos (UCI, licencia CC BY 4.0), junto a este cuaderno; su origen se explica en `LEEME.md`:
> Janosi, A., Steinbrunn, W., Pfisterer, M. y Detrano, R. (1989). *Heart Disease*. https://doi.org/10.24432/C52P4X ·
> Hofmann, H. (1994). *Statlog (German Credit Data)*. https://doi.org/10.24432/C5NC77 ·
> Sakar, C. O. y Kastro, Y. (2018). *Online Shoppers Purchasing Intention Dataset*. https://doi.org/10.24432/C5F88Q ·
> Chen, D. (2012). *Online Retail II*. https://doi.org/10.24432/C5CG6D

## 1. Caso de salud: enfermedad coronaria (2.1)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

d = pd.read_csv("corazon.csv")          # 303 pacientes de Cleveland
y = (d.pop("diagnostico") > 0).astype(int)   # 1 = enfermo
categoricas = ["sexo", "dolor", "glucosa", "ecg", "angina_esf",
               "pendiente", "talio"]
numericas = ["edad", "presion", "colesterol", "frec_max", "st_dep",
             "vasos"]

def preparacion():
    """Rellena los datos que faltan, codifica y escala."""
    cat = make_pipeline(SimpleImputer(strategy="most_frequent"),
                        OneHotEncoder(handle_unknown="ignore"))
    num = make_pipeline(SimpleImputer(strategy="median"),
                        StandardScaler())
    return make_column_transformer((cat, categoricas),
                                   (num, numericas))

X_ent, X_pru, y_ent, y_pru = train_test_split(
    d, y, test_size=0.3, stratify=y, random_state=0)

if __name__ == "__main__":
    print(d.shape, f"enfermos {y.mean():.1%}, datos que faltan "
          f"{d.isna().sum().sum()}")
    for nombre, m in [("referencia (clase mayoritaria)",
                       DummyClassifier()),
                      ("regresión logística", LogisticRegression())]:
        modelo = make_pipeline(preparacion(), m).fit(X_ent, y_ent)
        print(f"{nombre}: acierto en prueba "
              f"{modelo.score(X_pru, y_pru):.3f}")

## 2. Medidas de rendimiento y umbral (2.2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_score, StratifiedKFold

modelo = make_pipeline(preparacion(), LogisticRegression())
modelo.fit(X_ent, y_ent)
p = modelo.predict_proba(X_pru)[:, 1]        # probabilidad de enfermo

def medidas(umbral):
    vn, fp, fn, vp = confusion_matrix(y_pru, p >= umbral).ravel()
    return vp, fn, fp, vn, vp / (vp + fn), vn / (vn + fp)

for u in [0.5, 0.2]:
    vp, fn, fp, vn, sens, esp = medidas(u)
    print(f"umbral {u}: VP {vp} FN {fn} FP {fp} VN {vn} | "
          f"sensibilidad {sens:.3f}, especificidad {esp:.3f}")
print(f"área bajo la curva ROC: {roc_auc_score(y_pru, p):.3f}")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
acc = cross_val_score(modelo, d, y, cv=cv)     # 5 particiones
print("validación cruzada:", acc.round(3),
      f"media {acc.mean():.3f} ± {acc.std():.3f}")

fpr, tpr, umbrales = roc_curve(y_pru, p)
plt.plot(fpr, tpr, label="regresión logística")
plt.plot([0, 1], [0, 1], "--", lw=0.8, label="al azar")
for u in [0.5, 0.2]:
    _, _, _, _, s, e = medidas(u)
    plt.plot(1 - e, s, "o", ms=4)
    plt.annotate(f"umbral {u}", (1 - e, s), (6, -10),
                 textcoords="offset points")
plt.xlabel("1 - especificidad"); plt.ylabel("sensibilidad")
plt.legend(); plt.show()  # plt.savefig("roc.pdf")

**Prueba.** Busca el umbral más alto que mantenga la sensibilidad por encima de 0,95. ¿Cuántos sanos reciben una falsa alarma?

## 3. Caso de economía: crédito y matriz de costes (2.3)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

d = pd.read_csv("credito.csv")                 # 1000 clientes
y = (d.pop("riesgo") == 2).astype(int)         # 1 = mal pagador
cat = d.select_dtypes(exclude="number").columns  # A11, A12...
num = d.columns.difference(cat)
prep = make_column_transformer((OneHotEncoder(drop="first"), cat),
                               (StandardScaler(), num),
                               verbose_feature_names_out=False)
X_ent, X_pru, y_ent, y_pru = train_test_split(
    d, y, test_size=0.3, stratify=y, random_state=0)

# regularización L2: la validación cruzada elige su intensidad C
rejilla = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}
busca = GridSearchCV(make_pipeline(prep, LogisticRegression()),
                     rejilla, cv=5, scoring="roc_auc")
busca.fit(X_ent, y_ent)
modelo, lr = busca.best_estimator_, busca.best_estimator_[-1]
print(f"C elegido: {lr.C}; malos pagadores {y.mean():.0%}")
nombres = modelo[0].get_feature_names_out()
coef = pd.Series(lr.coef_[0], index=nombres).sort_values()
print("multiplican el riesgo (odds):")
print(np.exp(coef).round(2).iloc[[0, 1, 2, -3, -2, -1]])

# coste: aceptar a un mal pagador cuesta 5; rechazar a uno bueno, 1
p = modelo.predict_proba(X_pru)[:, 1]
def coste(rechazar):
    return 5 * np.sum(~rechazar & (y_pru == 1)) + \
           np.sum(rechazar & (y_pru == 0))
for nombre, r in [("aceptar a todos", np.zeros_like(p, bool)),
                  ("umbral 0.5", p >= 0.5),
                  ("umbral 1/6", p >= 1 / 6)]:
    print(f"{nombre:<16} coste {coste(r):4d}, rechaza {r.mean():.0%}")

**Prueba.** Quita las columnas `estado_sexo` y `extranjero` antes de entrenar (`d = d.drop(columns=[...])`). ¿Cambia mucho el coste?

## 4. Caso de empresa: previsión de ventas con un modelo lineal (2.3)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, r2_score,
                             root_mean_squared_error)

v = pd.read_csv("ventas_diarias.csv", parse_dates=["fecha"],
                index_col="fecha")["ventas"] / 1000  # miles de £

def variables(v):
    """Lo que se sabe la víspera: ventas pasadas y el calendario."""
    X = pd.DataFrame({"ayer": v.shift(1), "hace_7": v.shift(7),
                      "media_7": v.shift(1).rolling(7).mean(),
                      "media_28": v.shift(1).rolling(28).mean()})
    dias = pd.get_dummies(v.index.dayofweek, prefix="dia",
                          dtype=float).set_index(v.index)
    return X.join(dias).dropna()

X = variables(v)
y = v.loc[X.index]
ent, pru = X.index < "2011-07-01", X.index >= "2011-07-01"
print(f"entrenamiento {ent.sum()} días, prueba {pru.sum()} días")

def mide(nombre, pred):
    yp = y[pru]
    print(f"{nombre:<26} EAM {mean_absolute_error(yp, pred):5.2f}  "
          f"RECM {root_mean_squared_error(yp, pred):5.2f}  "
          f"R2 {r2_score(yp, pred):5.2f}")

if __name__ == "__main__":
    mide("referencia: hace 7 días", X.loc[pru, "hace_7"])
    lineal = LinearRegression().fit(X[ent], y[ent])
    mide("regresión lineal", lineal.predict(X[pru]))

**Ejercicio.** Añade la variable `hace_14` (las ventas de hace dos semanas). ¿Mejora la regresión lineal?

> Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, 27(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010